In [1]:
import torch
from torch import nn

In [2]:
x = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
target = 2 * x + 1

In [8]:
model = nn.Linear(1, 1)
with torch.no_grad():
    model.weight.fill_(0.0)
    model.bias.fill_(0.0)

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1, momentum=0.0, weight_decay=0.0
)
print("初始w/b:", model.weight.item(), model.bias.item())
print("学习率:", optimizer.param_groups[0]["lr"])

初始w/b: 0.0 0.0
学习率: 0.1


In [9]:
optimizer.zero_grad(set_to_none=True)
w_before = model.weight.item()
b_before = model.bias.item()

In [10]:
prediction = model(x)
loss = criterion(prediction, target)
print("预测:", prediction.detach())
print("loss:", loss.item())
print("backward前梯度:", model.weight.grad, model.bias.grad)

预测: tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.]])
loss: 9.0
backward前梯度: None None


In [11]:
loss.backward()

grad_w = model.weight.grad.item()
grad_b = model.bias.grad.item()
print("backward后梯度:", grad_w, grad_b)
print("backward后w/b:", model.weight.item(), model.bias.item())
print("原来的w/b:", w_before, b_before)

backward后梯度: -8.0 -2.0
backward后w/b: 0.0 0.0
原来的w/b: 0.0 0.0


In [12]:
lr = optimizer.param_groups[0]["lr"]

# 仅计算普通数字，还没有修改模型
expected_w = w_before - lr * grad_w
expected_b = b_before - lr * grad_b

optimizer.step()

actual_w = model.weight.item()
actual_b = model.bias.item()
print("公式预测w/b:", expected_w, expected_b)
print("step实际w/b:", actual_w, actual_b)
print("核对w:", abs(expected_w - actual_w) < 1e-6)
print("核对b:", abs(expected_b - actual_b) < 1e-6)
print("step后学习率:", optimizer.param_groups[0]["lr"])

with torch.no_grad():
    loss_after = criterion(model(x), target).item()
print("更新前/后loss:", loss.item(), loss_after)

公式预测w/b: 0.8 0.2
step实际w/b: 0.800000011920929 0.20000000298023224
核对w: True
核对b: True
step后学习率: 0.1
更新前/后loss: 9.0 3.5199997425079346
